In [ ]:
%%capture
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(quiet_load=False, dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from matplotlib.backends.backend_pdf import PdfPages
import configparser
import pandas as pd
import sqlalchemy

from clinicedc_constants import FEMALE, MALE
from clinicedc_constants import NO

In [ ]:
config = configparser.ConfigParser()
config.read(os.path.expanduser("~/.my.cnf"))

user = config["client"]["user"]
password = config["client"]["password"]
host = config["client"].get("host", "localhost")

meta2_engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/meta_production")
meta3_engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/meta3_production")


In [ ]:
scr_cols = [
    "screening_identifier",
    "subject_identifier",
    "site_id",
    "gender",
    "age_in_years",
    "converted_fbg_value",
    "converted_ogtt_value",
    "hba1c_value",
    "waist_circumference",
    "weight",
    "height",
    "dia_bp",
    "sys_bp",
    "bmi",
    "congestive_heart_failure",
    "liver_disease",
    "alcoholism",
    "acute_metabolic_acidosis",
    "renal_function_condition",
    "tissue_hypoxia_condition",
    "acute_condition",
    "metformin_sensitivity",
    "has_dm",
    "on_dm_medication",
    "unsuitable_for_study",
    "source"
]

In [ ]:
df_meta2_scr = pd.read_sql_table("meta_screening_subjectscreening", meta2_engine)
df_meta2_scr = df_meta2_scr.convert_dtypes()
df_meta2_scr = df_meta2_scr.rename(columns={"report_datetime": "screening_datetime", "calculated_bmi_value": "bmi"})
df_meta2_scr["dia_bp"] = (df_meta2_scr["dia_blood_pressure_one"] + df_meta2_scr["dia_blood_pressure_two"])/2
df_meta2_scr["sys_bp"] = (df_meta2_scr["sys_blood_pressure_one"] + df_meta2_scr["sys_blood_pressure_two"])/2
df_meta2_scr["source"] = "meta2"
len(df_meta2_scr[scr_cols])


In [ ]:
df_meta3_scr = pd.read_sql_table("meta_screening_subjectscreening", meta3_engine)
df_meta3_scr = df_meta3_scr[df_meta3_scr["meta_phase_two"]==NO]
df_meta3_scr = df_meta3_scr.convert_dtypes()
df_meta3_scr = df_meta3_scr.rename(columns={"report_datetime": "screening_datetime", "calculated_bmi_value": "bmi"})
df_meta3_scr["dia_bp"] = (df_meta3_scr["dia_blood_pressure_one"] + df_meta3_scr["dia_blood_pressure_two"])/2
df_meta3_scr["sys_bp"] = (df_meta3_scr["sys_blood_pressure_one"] + df_meta3_scr["sys_blood_pressure_two"])/2
df_meta3_scr["source"] = "meta3"
df_meta3_scr = df_meta3_scr.reset_index(drop=True)
len(df_meta3_scr[scr_cols])


In [ ]:
df = pd.concat([df_meta2_scr, df_meta3_scr], ignore_index=True)
df = df.convert_dtypes()
len(df)

In [ ]:
len(df[(df.hba1c_value>=6.5) & ((df.subject_identifier.str.startswith("101-")) | (df.subject_identifier.str.startswith("105-")))])

In [ ]:
df["hba1c"] = (df["hba1c_value"]>=6.5).astype("Int64")
df["fbg"] = (df["converted_fbg_value"]>=7.0).astype("Int64")
df["ogtt"] = (df["converted_ogtt_value"]>=11.1).astype("Int64")
df["hba1c_only"] = ((df["hba1c"]==1) & (df["fbg"]==0) & (df["ogtt"]==0)).astype("Int64")
df["fbg_only"] = ((df["hba1c"]==0) & (df["fbg"]==1) & (df["ogtt"]==0)).astype("Int64")
df["ogtt_only"] = ((df["hba1c"]==0) & (df["fbg"]==0) & (df["ogtt"]==1)).astype("Int64")

df["hba1c_fbg"] = ((df["hba1c"]==1) & (df["fbg"]==1) & (df["ogtt"]==0)).astype("Int64")
df["hba1c_ogtt"] = ((df["hba1c"]==1) & (df["fbg"]==0) & (df["ogtt"]==1)).astype("Int64")
df["fbg_ogtt"] = ((df["hba1c"]==0) & (df["fbg"]==1) & (df["ogtt"]==1)).astype("Int64")
df["hba1c_fbg_ogtt"] = ((df["hba1c"]==1) & (df["fbg"]==1) & (df["ogtt"]==1)).astype("Int64")
df["whtr"] = (df["waist_circumference"] / (df["height"]) >0.5).astype("Int64")

height_m = df["height"] / 100
bmi = df["weight"] / (height_m ** 2)
df["absi"] = df["waist_circumference"] / 100 / (bmi ** (2/3) * height_m ** (1/2))



In [ ]:
cond = ((df["hba1c"].isna()) | (df["fbg"].isna()) | (df["ogtt"].isna()))
df_subset = df[~cond][["gender", "age_in_years", "bmi", "weight", "height", "hba1c", "fbg", "ogtt", "hba1c_only", "fbg_only", "ogtt_only", "hba1c_fbg", "hba1c_ogtt", "fbg_ogtt", "hba1c_fbg_ogtt",  "whtr", "absi"]].reset_index(drop=True)
df_subset = df_subset.convert_dtypes()

df_subset.dtypes

In [ ]:
cols = ["hba1c", "fbg", "ogtt", "hba1c_only", "fbg_only", "ogtt_only", "hba1c_fbg", "hba1c_ogtt", "fbg_ogtt", "hba1c_fbg_ogtt", "whtr", "absi"]
df_subset[cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")

In [ ]:
cond = (df_subset["gender"]==FEMALE)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["gender"]==MALE)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["bmi"]<30.0)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["bmi"]>=30.0)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["age_in_years"]<50.0)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["age_in_years"]>=50.0)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
cond = (df_subset["whtr"]==1)
df_subset[cond][cols].sum().astype("Int64").rename_axis("measure").reset_index(name="total")


In [ ]:
df_subset

-- GROUP NAMES –
Group 1 Name: HbA1c
Group 2 Name: FBG
Group 3 Name: OGTT

-- DATA COUNTS –
Only Group 1: 457
Only Group 2: 587
Only Group 3: 64

Overlap of 1 & 2: 97
Overlap of 1 & 3: 39
Overlap of 2 & 3: 47


In [ ]:
# Center (Overlap of all 3): 139

# Script for the overall plot


In [ ]:
figures = []

In [ ]:
# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (457, 587, 97, 64, 39, 47, 139)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)

In [ ]:
# 1.	Script for the overall plot

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (457, 587, 97, 64, 39, 47, 139)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)


In [ ]:
# 2.	Script for the women only

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (391, 435, 84, 38, 25, 32, 95)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)
# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: Women", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)

In [ ]:
# 3.	Script for the men only

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (66, 152, 13, 26, 14, 15, 44)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: Men", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)


In [ ]:
# 4.	Script for the BMI<30

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (309, 412, 65, 41, 23, 25, 76)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: BMI<30", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)


In [ ]:
# 5.	Script for the BMI≥30

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (148, 175, 32, 23, 16, 22, 63)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: BMI≥30", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# 6.	Script for Age<50

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (294, 343, 56, 32, 12, 19, 55)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: Age <50 years", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
# plt.show()
plt.close(fig)

In [ ]:
# 7.	Script for Age ≥ 50 years

# 1. Input your exact mutually exclusive region totals
# Format sequence: (Only 1, Only 2, Overlap 1&2, Only 3, Overlap 1&3, Overlap 2&3, Center 1&2&3)
subsets_data = (163, 244, 41, 32, 27, 28, 84)

# 2. Initialize the plot
fig = plt.figure(figsize=(8, 8), dpi=300)
v = venn3(subsets=subsets_data, set_labels=('HbA1c', 'FBG', 'OGTT'))

# 3. Enhance text sizing and visibility
for label in v.set_labels:
    if label:
        label.set_fontsize(14)
for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

# 4. Set transparency across sections to keep numbers legible
for region_id in ['100', '010', '001', '110', '101', '011', '111']:
    patch = v.get_patch_by_id(region_id)
    if patch:
        patch.set_alpha(0.55)

# 5. Add title and display
plt.title("Blood Glucose Diagnostic Metric Overlap: Age ≥ 50 years", fontsize=16, fontweight='bold', pad=25)
figures.append(fig)
 #plt.show()
# plt.savefig('diagnostic_venn_diagram.png', bbox_inches='tight')
plt.close(fig)

In [ ]:
path = analysis_folder / "report.pdf"
with PdfPages(path) as pdf:
    for fig in figures:
        pdf.savefig(fig, bbox_inches="tight")